# Crop prices and deforestation: grid-cell regressions

> To write (Andrew): goal of the notebook. Points to cover: we estimate equation (3) of Berman et al.
> (2023); first we replicate their result (0.5 degree cells, 2001-2018, 15 crops), then we move to
> 9 km cells (same period), then we extend to 2025 (13 crops); what each step tells us.

## 1. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfixest as pf

plt.style.use("default")
plt.rcParams["figure.facecolor"] = "white"

# Data is read from GitHub. Switch BRANCH to "main" once the branch is merged.
BRANCH = "andrew-cleanup-berman"
BASE = f"https://raw.githubusercontent.com/AndrewEberhardt/DataScienceProject/{BRANCH}/data/"

# Same crops and price series as notebook 3
CROPS = {
    "banana": ["Banana, US"], "cocoa": ["Cocoa"], "coconut": ["Coconut oil"],
    "coffee": ["Coffee, Arabica", "Coffee, Robusta"], "cotton": ["Cotton, A Index"], "maize": ["Maize"],
    "oil palm": ["Palm oil"], "rice": ["Rice, Thai 5%"], "soybean": ["Soybeans"], "sugarcane": ["Sugar, world"],
    "tea": ["Tea, avg 3 auctions"], "tobacco": ["Tobacco, US import u.v."], "wheat": ["Wheat, US HRW"],
    "barley": ["Barley"], "sorghum": ["Sorghum"],
}
CROPS_13, CROPS_15 = list(CROPS)[:13], list(CROPS)
YEARS = list(range(2001, 2026))
BERMAN_BETA = 1.27     # Berman et al. (2023), Figure 1, Model 1 (se 0.09)
SOURCE = "Source: Hansen et al. GFC v1.13, FAO/IIASA GAEZ v4, World Bank Pink Sheet; own estimates."

## 2. Data

In [ ]:
t0 = time.time()
cells = pd.read_csv(BASE + "processed/grid_cells.csv.gz")
suit = pd.read_csv(BASE + "processed/grid_suitability.csv.gz")
suit.columns = ["cell_id"] + CROPS_15
cells = cells.merge(suit, on="cell_id", how="left")

prices = pd.read_csv(BASE + "processed/commodity_prices_annual.csv")
P = pd.DataFrame({c: prices[prices.commodity.isin(s)].groupby("year").price_real.mean() for c, s in CROPS.items()})
P = 100 * P / P.loc[2000]                      # base 100 in 2000, as Berman et al.

# 0.5 degree block of each 9 km cell (6 x 6 cells): replication unit and cluster
cells["block"] = (cells.row // 6) * 720 + cells.col // 6
print(f"{len(cells):,} cells, {cells.block.nunique():,} blocks of 0.5 degree, loaded in {time.time() - t0:.0f} s")

In [ ]:
def log_index(S, crops, years):
    """Log crop price index of Berman et al. (eq. 1-2): rows of S are units, columns are crops."""
    S = S[crops].to_numpy(float)
    tot = S.sum(axis=1, keepdims=True)
    alpha = np.divide(S, tot, out=np.full_like(S, np.nan), where=tot > 0)
    return np.log(alpha @ P.loc[years, crops].to_numpy().T)       # units x years


def to_long(units, id_col, loss_prefix, lnP, years, extra=()):
    """Long panel unit x year with the outcome (asinh of lost pixels), the loss count and the log index."""
    loss = units[[f"{loss_prefix}{y}" for y in years]].to_numpy()
    n, T = loss.shape
    d = pd.DataFrame({
        id_col: np.repeat(units[id_col].to_numpy(), T),
        "year": np.tile(years, n),
        "loss": loss.ravel(),
        "lnP": lnP.ravel().astype(np.float32),
    })
    for c in extra:
        d[c] = np.repeat(units[c].to_numpy(), T)
    # country x year fixed effect as one integer code (much lighter than iso3^year on millions of rows)
    country = pd.Categorical(units["iso3"]).codes.astype(np.int32)
    d["country"] = np.repeat(country, T)              # integer code, used to cluster by country
    d["ct"] = d.country * 10000 + d.year.to_numpy()
    d["y"] = np.arcsinh(d.loss).astype(np.float32)
    return d.dropna(subset=["lnP"])


def show(fit, name, keep="lnP"):
    t = fit.tidy().loc[[keep]] if keep else fit.tidy()
    t.insert(0, "model", name)
    t["N"] = fit._N
    return t

## 3. Replication of Berman et al. (2023): 0.5 degree cells, 2001-2018, 15 crops

We rebuild their unit: 0.5 x 0.5 degree cells in the tropics band. Forest and loss pixels are summed
over the 36 cells of 9 km; suitability is the mean over all land cells of the block (as an aggregation
of GAEZ to 0.5 degree). Country = country holding most of the block's forest. SE clustered by cell,
as in the paper.

In [ ]:
import rasterio

GAEZ_CODES = {"banana": ["ban"], "cocoa": ["coc"], "coconut": ["con"], "coffee": ["cof"], "cotton": ["cot"],
              "maize": ["mze"], "oil palm": ["olp"], "rice": ["rcd", "rcw"], "soybean": ["soy"],
              "sugarcane": ["suc"], "tea": ["tea"], "tobacco": ["tob"], "wheat": ["whe"],
              "barley": ["brl"], "sorghum": ["srg"]}

def block_mean_suitability():
    """Mean GAEZ suitability over the land cells of each 0.5 degree block."""
    out = {}
    for crop, codes in GAEZ_CODES.items():
        layers = []
        for c in codes:
            with rasterio.open(BASE + f"raw/gaez/sxHr_{c}.tif") as src:
                layers.append(src.read(1).astype(np.float32))
        a = np.max(layers, axis=0)
        a[a < 0] = np.nan                                     # water / no data
        out[crop] = np.nanmean(a.reshape(360, 6, 720, 6), axis=(1, 3)).ravel() / 10000
    return pd.DataFrame(out).rename_axis("block").reset_index()

def make_blocks(prefix):
    """Aggregate 9 km cells to 0.5 degree blocks (pixel counts summed)."""
    cols = [f"{prefix}{y}" for y in YEARS] + ["forest25_px", "forest50_px"]
    b = cells.groupby("block")[cols].sum().reset_index()
    main = (cells.groupby(["block", "iso3"]).forest25_px.sum().reset_index()
            .sort_values("forest25_px").drop_duplicates("block", keep="last")[["block", "iso3"]])
    b = b.merge(main, on="block")
    lat = 90 - ((b.block // 720) * 6 + 3) / 12
    b["in_tropics"] = (lat.abs() <= 23.44).astype(int)
    b["lat"] = lat
    b["lon"] = -180 + ((b.block % 720) * 6 + 3) / 12
    return b.merge(block_mean_suitability(), on="block", how="left")

blocks = make_blocks("loss25_")
bt = blocks[blocks.in_tropics == 1].reset_index(drop=True)
print(f"0.5 degree cells in the tropics band with forest: {len(bt):,} (Berman et al.: 12,288 over the whole band)")

In [ ]:
YEARS_R = list(range(2001, 2019))
rep = to_long(bt, "block", "loss25_", log_index(bt, CROPS_15, YEARS_R), YEARS_R, extra=("lat", "lon"))
fit_rep = pf.feols("y ~ lnP | block + ct", rep, vcov={"CRV1": "block"})
res = [show(fit_rep, "Replication: 0.5 deg, 2001-2018, 15 crops")]
res[-1]

Model 2 of Berman et al.: the effect varies across deciles of forest cover in 2000
(deforestation is bounded by the initial forest).

In [ ]:
bt["decile"] = pd.qcut(bt.forest25_px, 10, labels=False) + 1
rep["decile"] = rep.block.map(bt.set_index("block").decile)
fit_rep2 = pf.feols("y ~ i(decile, lnP) | block + ct", rep, vcov={"CRV1": "block"})

def plot_deciles(fits, labels, title, fname):
    fig, ax = plt.subplots(figsize=(8, 5))
    for k, (fit, lab) in enumerate(zip(fits, labels)):
        t = fit.tidy()
        x = np.arange(1, 11) + (k - (len(fits) - 1) / 2) * 0.15
        ax.errorbar(x, t.Estimate, yerr=1.96 * t["Std. Error"], fmt="o", capsize=3, label=lab)
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_xticks(range(1, 11))
    ax.set_xlabel("Decile of forest cover in 2000")
    ax.set_ylabel("Effect of log price index (95% CI)")
    ax.set_title(title)
    ax.legend()
    fig.text(0.01, -0.03, SOURCE, fontsize=9)
    fig.savefig(f"../figures/{fname}", dpi=200, bbox_inches="tight")
    plt.show()

plot_deciles([fit_rep2], ["0.5 deg, 2001-2018, 15 crops"], "Effect of crop prices by initial forest cover (replication)",
             "replication_deciles.png")

## 4. Finer: 9 km cells, 2001-2018, 15 crops

Same period and crops as the replication. SE clustered by 0.5 degree block (neighbouring 9 km cells are
strongly correlated); clustering by cell shown for comparison.

In [ ]:
fine = to_long(cells, "cell_id", "loss25_", log_index(cells, CROPS_15, YEARS_R), YEARS_R, extra=("block",))
fit_fine = pf.feols("y ~ lnP | cell_id + ct", fine, vcov={"CRV1": "block"}, copy_data=False)
res.append(show(fit_fine, "9 km, 2001-2018, 15 crops (cluster 0.5 deg)"))
se_cell = pf.feols("y ~ lnP | cell_id + ct", fine, vcov={"CRV1": "cell_id"}, copy_data=False, lean=True)
res.append(show(se_cell, "same, SE clustered by 9 km cell"))
pd.concat(res[-2:])

## 5. Longer: 9 km cells, 2001-2025, 13 crops

Main specification of our article. Then: does the effect differ after 2018?

In [ ]:
del fine
main = to_long(cells, "cell_id", "loss25_", log_index(cells, CROPS_13, YEARS), YEARS, extra=("block",))
main["post2018"] = (main.year > 2018).astype(int)
fit_main = pf.feols("y ~ lnP | cell_id + ct", main, vcov={"CRV1": "block"}, copy_data=False)
res.append(show(fit_main, "MAIN: 9 km, 2001-2025, 13 crops"))
fit_post = pf.feols("y ~ lnP + lnP:post2018 | cell_id + ct", main, vcov={"CRV1": "block"}, copy_data=False, lean=True)
res.append(show(fit_post, "9 km, 2001-2025: extra effect after 2018", keep="lnP:post2018"))
pd.concat(res[-2:])

In [ ]:
# Same 13 crops and period at 0.5 degree, to separate the effect of resolution from the effect of period
fit_b13 = pf.feols("y ~ lnP | block + ct",
                   (rep13 := to_long(bt, "block", "loss25_", log_index(bt, CROPS_13, YEARS), YEARS, extra=("lat", "lon"))),
                   vcov={"CRV1": "block"}, lean=True)
res.append(show(fit_b13, "0.5 deg, 2001-2025, 13 crops"))
res[-1]

## 6. Contribution of crop prices to deforestation

As Berman et al. (footnote 10): predicted deforestation with observed prices vs prices fixed at their
2001 level, using the decile model. Predictions are transformed back from asinh with sinh.

In [ ]:
cell_dec = pd.qcut(cells.forest25_px, 10, labels=False) + 1
main["decile"] = np.repeat(cell_dec.to_numpy(), len(YEARS))[main.index]
fit_main2 = pf.feols("y ~ i(decile, lnP) | cell_id + ct", main, vcov={"CRV1": "block"}, copy_data=False)
beta_d = fit_main2.coef().to_numpy()
pred = fit_main2.predict()
assert len(pred) == len(main), "some observations were dropped by the estimation"
lnP2001 = main.groupby("cell_id").lnP.transform("first").to_numpy()   # first year = 2001
shift = beta_d[main.decile.to_numpy() - 1] * (main.lnP.to_numpy() - lnP2001)
obs, cf = np.sinh(pred), np.sinh(pred - shift)
contrib = (obs.sum() - cf.sum()) / cf.sum()
print(f"Crop price changes since 2001 explain {contrib:.0%} of predicted deforestation, 2001-2025 "
      f"(Berman et al.: 35% over 2001-2018)")
plot_deciles([fit_rep2, fit_main2], ["0.5 deg, 2001-2018, 15 crops", "9 km, 2001-2025, 13 crops"],
             "Effect of crop prices by initial forest cover", "main_deciles.png")

## 7. Inference: how precise is beta?

Three ways to compute the standard error of the main coefficients, from the least to the most conservative
assumption about correlated errors, plus a permutation test.
- Cluster by 0.5 degree block (main, as Berman et al.).
- Cluster by country (62 clusters): any correlation within a country is allowed.
- Conley (500 and 1,000 km): correlation between all cells closer than the cutoff in the same year, and
  within a cell over time (Berman et al., Tables OA.13-OA.14). Computed at 0.5 degree only: at 9 km the
  number of cell pairs is too large.
- Permutation of prices across crops: all shocks come from a few world prices, so cells with a similar
  crop mix receive the same shocks wherever they are (Adao, Kolesar and Morales 2019). We shuffle the
  price series between crops, re-estimate beta, and compare the true beta with this distribution.

In [ ]:
from scipy.spatial import cKDTree

def demean(v, groups, n_iter=500, tol=1e-9):
    """Remove several sets of fixed effects by alternating projections."""
    v = v.astype(float).copy()
    for _ in range(n_iter):
        old = v.copy()
        for g in groups:
            v -= pd.Series(v).groupby(g).transform("mean").to_numpy()
        if np.abs(v - old).max() < tol:
            break
    return v


def conley_se(d, unit, cutoff_km, x="lnP", y="y"):
    """Conley SE of y ~ x | unit + ct. Uniform kernel in space (same year, distance < cutoff)
    plus serial correlation within each unit over all years."""
    groups = [d[unit].to_numpy(), d["ct"].to_numpy()]
    xt, yt = demean(d[x].to_numpy(), groups), demean(d[y].to_numpy(), groups)
    b = (xt * yt).sum() / (xt ** 2).sum()
    score = xt * (yt - b * xt)
    u = d[[unit, "lat", "lon"]].drop_duplicates(unit).reset_index(drop=True)
    la, lo = np.radians(u.lat.to_numpy()), np.radians(u.lon.to_numpy())
    xyz = np.c_[np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)]
    pairs = cKDTree(xyz).query_pairs(2 * np.sin(cutoff_km / 6371 / 2), output_type="ndarray")
    pos = pd.Series(np.arange(len(u)), index=u[unit])
    S = np.zeros((len(u), d.year.nunique()))
    np.add.at(S, (pos[d[unit]].to_numpy(), pd.Categorical(d.year).codes), score)
    spatial = (S ** 2).sum() + 2 * (S[pairs[:, 0]] * S[pairs[:, 1]]).sum()
    serial = (S.sum(axis=1) ** 2).sum() - (S ** 2).sum()
    return b, np.sqrt(spatial + serial) / (xt ** 2).sum()


def permutation_test(units, d, unit, crops, years, n_perm=500, seed=0):
    """Re-estimate beta after shuffling the price series across crops."""
    rng = np.random.default_rng(seed)
    S = units[crops].to_numpy(float)
    tot = S.sum(axis=1, keepdims=True)
    alpha = np.divide(S, tot, out=np.full_like(S, np.nan), where=tot > 0)
    keep = units[unit].isin(d[unit]).to_numpy()
    prices = P.loc[years, crops].to_numpy()
    groups = [d[unit].to_numpy(), d["ct"].to_numpy()]
    yt = demean(d.y.to_numpy(), groups)
    betas = []
    for _ in range(n_perm):
        lnp = np.log(alpha[keep] @ prices[:, rng.permutation(len(crops))].T).ravel()
        xt = demean(lnp[~np.isnan(lnp)], groups)
        betas.append((xt * yt).sum() / (xt ** 2).sum())
    return np.array(betas)

In [ ]:
inf = []
specs = [("0.5 deg, 2001-2018, 15 crops (replication)", rep, "block", bt, CROPS_15, YEARS_R),
         ("0.5 deg, 2001-2025, 13 crops", rep13, "block", bt, CROPS_13, YEARS)]
for name, d, unit, units, crops, years in specs:
    t0 = time.time()
    row = {"model": name}
    for label, vc in [("SE cluster 0.5 deg", {"CRV1": "block"}), ("SE cluster country", {"CRV1": "country"})]:
        fit = pf.feols("y ~ lnP | block + ct", d, vcov=vc, lean=True)
        row["beta"] = fit.coef().iloc[0]
        row[label] = fit.se().iloc[0]
    for km in (500, 1000):
        row[f"SE Conley {km} km"] = conley_se(d, unit, km)[1]
    perm = permutation_test(units, d, unit, crops, years)
    row["permutation p"] = (np.abs(perm) >= abs(row["beta"])).mean()
    row["permutation 95th pct |beta|"] = np.percentile(np.abs(perm), 95)
    inf.append(row)
    print(f"{name}: {time.time() - t0:.0f} s")

# Main 9 km specification: block and country clusters (Conley and permutation too heavy at 9 km)
for label, vc in [("SE cluster 0.5 deg", {"CRV1": "block"}), ("SE cluster country", {"CRV1": "country"})]:
    fit = pf.feols("y ~ lnP | cell_id + ct", main, vcov=vc, copy_data=False, lean=True)
    inf.append({"model": "MAIN: 9 km, 2001-2025, 13 crops", "beta": fit.coef().iloc[0], label: fit.se().iloc[0]})
inference = pd.DataFrame(inf).groupby("model", sort=False).first()
inference.round(4)

## 8. Robustness

In [ ]:
rob = []
# Canopy threshold 50 percent
m50 = to_long(cells, "cell_id", "loss50_", log_index(cells, CROPS_13, YEARS), YEARS, extra=("block",))
rob.append(show(pf.feols("y ~ lnP | cell_id + ct", m50, vcov={"CRV1": "block"}, copy_data=False, lean=True),
                "Canopy 50%"))
del m50
# Lagged price index
main = main.sort_values(["cell_id", "year"])
for k in (1, 2):
    main[f"lnP_l{k}"] = main.groupby("cell_id").lnP.shift(k)
    rob.append(show(pf.feols(f"y ~ lnP_l{k} | cell_id + ct", main.dropna(subset=[f"lnP_l{k}"]),
                             vcov={"CRV1": "block"}, lean=True), f"Price index t-{k}", keep=f"lnP_l{k}"))
# Average suitability x year (Berman et al., Figure OA.12)
main["avg_suit"] = np.repeat(cells[CROPS_13].mean(axis=1).to_numpy(), len(YEARS))[main.index]
rob.append(show(pf.feols("y ~ lnP + i(year, avg_suit) | cell_id + ct", main, vcov={"CRV1": "block"},
                         copy_data=False, lean=True), "Average suitability x year"))
# Tropics band only
rob.append(show(pf.feols("y ~ lnP | cell_id + ct", main[main.cell_id.isin(cells.cell_id[cells.in_tropics == 1])],
                         vcov={"CRV1": "block"}, lean=True), "Tropics band only"))
pd.concat(rob)

In [ ]:
# PPML on pixel counts: coefficients are elasticities, comparable across resolutions
for name, d, fe in [("PPML replication: 0.5 deg, 2001-2018, 15 crops", rep, "block"),
                    ("PPML main: 9 km, 2001-2025, 13 crops", main, "cell_id")]:
    fit = pf.fepois(f"loss ~ lnP | {fe} + ct", d, vcov={"CRV1": "block"}, lean=True)
    rob.append(show(fit, name))
pd.concat(rob[-2:])

## 9. Summary

In [ ]:
summary = pd.concat(res + rob)[["model", "Estimate", "Std. Error", "Pr(>|t|)", "2.5%", "97.5%", "N"]]
summary.round(3).reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.45 * len(summary) + 1))
y = np.arange(len(summary))[::-1]
ax.errorbar(summary.Estimate, y, xerr=1.96 * summary["Std. Error"], fmt="o", capsize=3)
ax.axvline(0, color="grey", lw=0.6)
ax.axvline(BERMAN_BETA, color="tab:red", ls="--", lw=1, label="Berman et al. (2023): 1.27")
ax.set_yticks(y)
ax.set_yticklabels(summary.model)
ax.set_xlabel("Effect of log crop price index on asinh(deforested pixels), 95% CI")
ax.set_title("Crop prices and deforestation: all specifications")
ax.legend(loc="lower right")
fig.text(0.01, -0.02, SOURCE, fontsize=9)
fig.savefig("../figures/grid_coefficients.png", dpi=200, bbox_inches="tight")
plt.show()